# 01. vLLM 기본 사용법

이 노트북에서는 vLLM의 기본 개념과 사용법을 학습합니다.

## 목차
1. vLLM 소개
2. 기본 추론
3. Sampling Parameters
4. 배치 추론
5. 스트리밍 출력

## 1. vLLM 소개

vLLM은 고성능 LLM 추론 및 서빙 라이브러리입니다.

### 핵심 기능
- **Continuous Batching**: 동적 배치 처리로 처리량 극대화
- **PagedAttention**: 효율적인 메모리 관리
- **다양한 양자화 지원**: AWQ, GPTQ, INT8 등
- **OpenAI 호환 API**: 기존 코드 마이그레이션 용이

In [ ]:
# 필요한 라이브러리 임포트
import os
import json
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv('../.env')

print("vLLM 학습 노트북 시작!")

In [ ]:
# vLLM 설치 확인
try:
    import vllm
    print(f"vLLM version: {vllm.__version__}")
except ImportError:
    print("vLLM이 설치되어 있지 않습니다.")
    print("설치 명령어: pip install vllm")

In [ ]:
# GPU 확인
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. 기본 추론

vLLM의 `LLM` 클래스를 사용한 기본 추론 방법을 알아봅니다.

In [ ]:
# 모델 로드 (GPU 필요)
# AWQ 양자화 모델 사용

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-AWQ"

# vLLM이 설치되어 있고 GPU가 있을 때만 실행
if 'vllm' in dir() and torch.cuda.is_available():
    from vllm import LLM, SamplingParams
    
    print(f"모델 로딩 중: {MODEL_NAME}")
    print("(처음 실행 시 모델 다운로드에 시간이 걸릴 수 있습니다)")
    
    # LLM 인스턴스 생성
    llm = LLM(
        model=MODEL_NAME,
        quantization="awq",
        gpu_memory_utilization=0.90,
        trust_remote_code=True
    )
    
    print("모델 로드 완료!")
else:
    print("vLLM 또는 GPU를 사용할 수 없습니다.")
    print("아래 셀들은 예시 코드로 참고하세요.")
    llm = None

In [ ]:
# 기본 추론 예제

def run_inference(prompt, max_tokens=256, temperature=0.7):
    """기본 추론 실행"""
    if llm is None:
        print(f"[예시] 프롬프트: {prompt[:50]}...")
        print("[예시] 응답: (모델 로드 필요)")
        return None
    
    sampling_params = SamplingParams(
        temperature=temperature,
        max_tokens=max_tokens
    )
    
    outputs = llm.generate([prompt], sampling_params)
    return outputs[0].outputs[0].text

# 테스트
prompt = """<|im_start|>system
당신은 친절한 AI 어시스턴트입니다.<|im_end|>
<|im_start|>user
안녕하세요! vLLM에 대해 간단히 설명해주세요.<|im_end|>
<|im_start|>assistant
"""

response = run_inference(prompt)
if response:
    print(f"응답:\n{response}")

## 3. Sampling Parameters

vLLM의 다양한 샘플링 파라미터를 알아봅니다.

In [ ]:
# SamplingParams 옵션

sampling_params_example = {
    "temperature": 0.7,       # 샘플링 온도 (0.0~2.0)
    "top_p": 0.9,             # Top-p (nucleus) 샘플링
    "top_k": 50,              # Top-k 샘플링
    "max_tokens": 512,        # 최대 생성 토큰 수
    "repetition_penalty": 1.0, # 반복 패널티
    "stop": ["<|im_end|>"],   # 중단 토큰
}

print("주요 SamplingParams 옵션:")
for key, value in sampling_params_example.items():
    print(f"  {key}: {value}")

In [ ]:
# 온도에 따른 출력 변화 실험

test_prompt = """<|im_start|>system
당신은 창의적인 작가입니다.<|im_end|>
<|im_start|>user
가을에 대한 한 줄 시를 써주세요.<|im_end|>
<|im_start|>assistant
"""

temperatures = [0.1, 0.5, 1.0, 1.5]

print("온도별 출력 비교:")
print("=" * 60)

for temp in temperatures:
    print(f"\nTemperature = {temp}:")
    if llm is not None:
        sampling_params = SamplingParams(temperature=temp, max_tokens=64)
        outputs = llm.generate([test_prompt], sampling_params)
        print(f"  {outputs[0].outputs[0].text.strip()}")
    else:
        print("  (모델 로드 필요)")

## 4. 배치 추론

여러 프롬프트를 한 번에 처리하는 배치 추론을 알아봅니다.

In [ ]:
# 배치 추론 예제

batch_prompts = [
    "서울의 인구는 몇 명인가요?",
    "파이썬의 장점을 설명해주세요.",
    "오늘 날씨가 좋네요.",
]

def format_prompt(message):
    return f"""<|im_start|>system
당신은 친절한 AI 어시스턴트입니다.<|im_end|>
<|im_start|>user
{message}<|im_end|>
<|im_start|>assistant
"""

formatted_prompts = [format_prompt(p) for p in batch_prompts]

print(f"배치 크기: {len(formatted_prompts)}")

if llm is not None:
    import time
    
    start_time = time.time()
    sampling_params = SamplingParams(temperature=0.7, max_tokens=128)
    outputs = llm.generate(formatted_prompts, sampling_params)
    elapsed = time.time() - start_time
    
    print(f"\n처리 시간: {elapsed:.2f}초")
    print(f"평균 시간/요청: {elapsed/len(batch_prompts):.2f}초")
    
    print("\n결과:")
    for i, (prompt, output) in enumerate(zip(batch_prompts, outputs)):
        print(f"\n[{i+1}] Q: {prompt}")
        print(f"    A: {output.outputs[0].text.strip()[:100]}...")
else:
    print("(모델 로드 필요)")

## 5. 스트리밍 출력

AsyncLLMEngine을 사용한 스트리밍 출력 방법을 알아봅니다.

In [ ]:
# 스트리밍 추론 예제 (AsyncLLMEngine 사용)

streaming_code_example = '''
# AsyncLLMEngine을 사용한 스트리밍 예제

import asyncio
from vllm import AsyncLLMEngine, SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs

async def stream_generate(prompt):
    # 엔진 초기화
    engine_args = AsyncEngineArgs(
        model="Qwen/Qwen2.5-7B-Instruct-AWQ",
        quantization="awq"
    )
    engine = AsyncLLMEngine.from_engine_args(engine_args)
    
    sampling_params = SamplingParams(
        temperature=0.7,
        max_tokens=256
    )
    
    # 스트리밍 생성
    request_id = "stream_001"
    results_generator = engine.generate(prompt, sampling_params, request_id)
    
    previous_text = ""
    async for request_output in results_generator:
        output = request_output.outputs[0]
        new_text = output.text[len(previous_text):]
        previous_text = output.text
        
        # 새로 생성된 텍스트 출력
        print(new_text, end="", flush=True)
    
    print()  # 줄바꿈

# 실행
asyncio.run(stream_generate(prompt))
'''

print("스트리밍 추론 코드 예제:")
print("=" * 60)
print(streaming_code_example)

## 요약

### 학습 내용
1. **vLLM 기본 개념**: Continuous Batching, PagedAttention
2. **LLM 클래스**: 기본 추론 방법
3. **SamplingParams**: temperature, top_p, top_k, max_tokens 등
4. **배치 추론**: 여러 프롬프트 동시 처리
5. **스트리밍**: AsyncLLMEngine 사용

### 다음 단계
- `02_performance_tuning.ipynb`: 성능 튜닝 실험
- `03_comparison.ipynb`: vLLM vs HuggingFace 비교